In [8]:
# Do not run - This script was use dto create the 3d depth maps.

import torch
from transformers import pipeline, logging
from PIL import Image

logging.set_verbosity_error()

depth_estimator = pipeline(
    'depth-estimation',
    model='Intel/dpt-large',
    device=0
)


def depthMaps(ipath, opath):

    img = Image.open(ipath)

    predictions = depth_estimator(img)
    depth_map = predictions['depth']

    depth_map.save(opath)

import os
ipath = "/content/drive/MyDrive/CIS_5190_Project_Material/Images"
opath = "/content/drive/MyDrive/CIS_5190_Project_Material/Depth_Maps"
os.makedirs(opath, exist_ok=True)

for filename in os.listdir(ipath):
  if filename.endswith(".jpg"):
    ipath_file = os.path.join(ipath, filename)
    opath_file = os.path.join(opath, filename)
    depthMaps(ipath_file, opath_file)


config.json:   0%|          | 0.00/942 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.37G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import os
from google.colab import userdata
from huggingface_hub import login
from diffusers import (
    StableDiffusionControlNetPipeline,
    ControlNetModel,
    MultiControlNetModel,
    DDPMScheduler,
    AutoencoderKL,
    UNet2DConditionModel
)
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import CLIPTokenizer, CLIPTextModel
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [11]:
class PennCampusDataset(Dataset):
    def __init__(self, csv_path, image_dir, edge_dir, depth_dir, tokenizer, resolution=512): # Added depth_dir
        self.df = pd.read_csv(csv_path)
        self.df = self._validate_files(self.df, image_dir, edge_dir, depth_dir) # Passed depth_dir

        self.image_dir = image_dir
        self.edge_dir = edge_dir
        self.depth_dir = depth_dir # Saved depth_dir
        self.tokenizer = tokenizer
        self.resolution = resolution

        self.image_transform = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ])

        self.edge_transform = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
        ])

        print(f"Dataset initialized with {len(self.df)} valid samples")

    def _validate_files(self, df, image_dir, edge_dir, depth_dir):
        valid_rows = []
        missing_count = 0

        for idx, row in df.iterrows():
            filename = row['filename']
            image_path = os.path.join(image_dir, filename)
            edge_path = os.path.join(edge_dir, filename)
            depth_path = os.path.join(depth_dir, filename) # New check

            if os.path.exists(image_path) and os.path.exists(edge_path) and os.path.exists(depth_path):
                valid_rows.append(row)
            else:
                missing_count += 1
                print(f"Skipping missing file: {filename}")

        if missing_count > 0:
            print(f"Warning: {missing_count} files missing and skipped")

        return pd.DataFrame(valid_rows).reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['filename']
        prompt = row['prompt']

        # Load all three images
        image_path = os.path.join(self.image_dir, filename)
        image = Image.open(image_path).convert("RGB")

        edge_path = os.path.join(self.edge_dir, filename)
        edge = Image.open(edge_path).convert("RGB")

        depth_path = os.path.join(self.depth_dir, filename) # New depth load
        depth = Image.open(depth_path).convert("RGB")

        # Apply transforms
        pixel_values = self.image_transform(image)
        canny_pixel_values = self.edge_transform(edge) # Renamed
        depth_pixel_values = self.edge_transform(depth) # New depth transform

        tokenized = self.tokenizer(
            prompt,
            max_length=self.tokenizer.model_max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "pixel_values": pixel_values,
            "canny_pixel_values": canny_pixel_values, # Split into two explicit keys
            "depth_pixel_values": depth_pixel_values,
            "input_ids": tokenized.input_ids.squeeze(0),
            "prompt": prompt,
        }

# Update the collate function to match the new keys
def collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "canny_pixel_values": torch.stack([b["canny_pixel_values"] for b in batch]),
        "depth_pixel_values": torch.stack([b["depth_pixel_values"] for b in batch]),
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
    }

In [12]:
def compute_validation_loss(unet, controlnet, val_loader, noise_scheduler, vae, text_encoder, device, dtype):
    """Compute validation loss on held-out dataset."""
    unet.eval()
    total_loss = 0
    num_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            # 1. NEW BATCH KEYS: Load both Canny and Depth maps
            pixel_values = batch["pixel_values"].to(device, dtype=dtype, non_blocking=True)
            canny_cond = batch["canny_pixel_values"].to(device, dtype=dtype, non_blocking=True)
            depth_cond = batch["depth_pixel_values"].to(device, dtype=dtype, non_blocking=True)
            input_ids = batch["input_ids"].to(device, non_blocking=True)

            # Encode to latents
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

            # Add noise
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps,
                (latents.shape[0],), device=device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # Get text embeddings
            encoder_hidden_states = text_encoder(input_ids)[0]

            # 2. MIXED PRECISION: Keep it fast on the A100
            with torch.autocast("cuda", dtype=torch.bfloat16):
                # 3. LIST INJECTION: Pass both conditions to MultiControlNet
                down_block_res_samples, mid_block_res_sample = controlnet(
                    noisy_latents, timesteps, encoder_hidden_states,
                    controlnet_cond=[canny_cond, depth_cond],
                    conditioning_scale=[1.0,1.0],
                    return_dict=False,
                )

                # UNet forward
                noise_pred = unet(
                    noisy_latents, timesteps, encoder_hidden_states,
                    down_block_additional_residuals=down_block_res_samples,
                    mid_block_additional_residual=mid_block_res_sample,
                ).sample

                # 4. INTACT LOSS: The math doesn't change
                loss = F.mse_loss(noise_pred.float(), noise.float())

            total_loss += loss.item()
            num_batches += 1

    avg_val_loss = total_loss / num_batches
    unet.train()
    return avg_val_loss

In [13]:
# Optional: Visual validation - generates sample images
VISUAL_VALIDATION_SAMPLES = [
    {
        "edge_map": "/content/drive/MyDrive/CIS_5190_Project_Material/CannyMap/vp_sunny_day_front_facing.jpg",
        "depth_map": "/content/drive/MyDrive/CIS_5190_Project_Material/Depth_Maps/vp_sunny_day_front_facing.jpg", # NEW: Added depth map path
        "prompt": "PENNVP, Van Pelt library, modernist brick and concrete facade, large rectangular windows, flat roofline, five-story library building, front facing view, night time, dark clear sky, warm lamppost glow, artificial lighting, illuminated windows, lit facade, Penn campus, University of Pennsylvania, Philadelphia, architectural photography, Upenn",
        "name": "vp_sunny_to_night"
    },
    {
        "edge_map": "/content/drive/MyDrive/CIS_5190_Project_Material/CannyMap/collegehall_sunny_day_front_facing_right.jpg",
        "depth_map": "/content/drive/MyDrive/CIS_5190_Project_Material/Depth_Maps/collegehall_sunny_day_front_facing_right.jpg", # NEW: Added depth map path
        "prompt": "PENNCH, College Hall, gothic collegiate architecture, green serpentine stone facade, rainy day, wet pavement, puddle reflections, overcast grey sky, Penn campus, University of Pennsylvania, Philadelphia, architectural photography, Upenn",
        "name": "ch_sunny_to_rainy"
    },
]

def generate_visual_validation(unet, controlnet, epoch, output_dir):
    """Generate visual validation samples."""
    print(f"  Generating visual samples...")

    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "/content/drive/MyDrive/CIS_5190_Project_Material/Models/sd-v1-5",
        controlnet=controlnet,
        torch_dtype=torch.bfloat16,
        local_files_only=True
    ).to("cuda")

    pipe.unet = unet
    pipe.set_progress_bar_config(disable=True)

    unet.eval()

    for sample in VISUAL_VALIDATION_SAMPLES:
        # Load BOTH condition images
        edge_map = Image.open(sample["edge_map"]).convert("RGB").resize((512, 512))
        depth_map = Image.open(sample["depth_map"]).convert("RGB").resize((512, 512)) # NEW: Load depth map

        # Generate using the bundled Multi-ControlNet
        generated = pipe(
            prompt=sample["prompt"],
            image=[edge_map, depth_map], # NEW: Pass as a list! [Canny, Depth]
            num_inference_steps=30,
            guidance_scale=7.5,
            controlnet_conditioning_scale=[0.6, 0.4], # NEW: Weight balancing [Canny=60%, Depth=40%]
        ).images[0]

        output_path = os.path.join(output_dir, f"epoch_{epoch}_{sample['name']}.png")
        generated.save(output_path)

    unet.train()
    del pipe
    torch.cuda.empty_cache()

In [14]:
# Training configuration
MODEL_PATH = "/content/drive/MyDrive/CIS_5190_Project_Material/Models/sd-v1-5"

# --- NEW: Split the ControlNet paths! ---
CONTROLNET_CANNY_PATH = "/content/drive/MyDrive/CIS_5190_Project_Material/Models/controlnet-canny"
CONTROLNET_DEPTH_PATH = "/content/drive/MyDrive/CIS_5190_Project_Material/Models/controlnet-depth"
# ----------------------------------------

OUTPUT_DIR = "/content/drive/MyDrive/CIS_5190_Project_Material/LoRA_Checkpoints(D+C)"
VALIDATION_DIR = "/content/drive/MyDrive/CIS_5190_Project_Material/Validation_Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(VALIDATION_DIR, exist_ok=True)

# Hyperparameters
RESOLUTION = 512
BATCH_SIZE = 4  # Reduce to 2 if OOM
LEARNING_RATE = 1e-4
NUM_EPOCHS = 15
SAVE_EVERY_N_EPOCH = 5
VALIDATE_EVERY_N_EPOCHS = 5
LORA_RANK = 8
LORA_ALPHA = 32
MAX_GRAD_NORM = 1.0

device = "cuda"
dtype = torch.bfloat16  # Consistent bfloat16 everywhere

In [15]:
# Load model components
print("Loading model components...")

tokenizer = CLIPTokenizer.from_pretrained(MODEL_PATH, subfolder="tokenizer", local_files_only=True)
text_encoder = CLIPTextModel.from_pretrained(
    MODEL_PATH, subfolder="text_encoder", torch_dtype=dtype, local_files_only=True
).to(device)
vae = AutoencoderKL.from_pretrained(
    MODEL_PATH, subfolder="vae", torch_dtype=dtype, local_files_only=True
).to(device)
unet = UNet2DConditionModel.from_pretrained(
    MODEL_PATH, subfolder="unet", torch_dtype=dtype, local_files_only=True
).to(device)
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_PATH, subfolder="scheduler", local_files_only=True)

# --- NEW: Load BOTH ControlNets and bundle them ---
print("Loading Multi-ControlNet...")
controlnet_canny = ControlNetModel.from_pretrained(
    CONTROLNET_CANNY_PATH, torch_dtype=dtype, local_files_only=True
)
controlnet_depth = ControlNetModel.from_pretrained(
    CONTROLNET_DEPTH_PATH, torch_dtype=dtype, local_files_only=True
)
controlnet = MultiControlNetModel([controlnet_canny, controlnet_depth]).to(device)
# --------------------------------------------------

# Freeze components
text_encoder.requires_grad_(False)
vae.requires_grad_(False)
unet.requires_grad_(False)
controlnet.requires_grad_(False)

print("✓ Models loaded")

# Inject LoRA into UNet
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
    lora_dropout=0.1,
    bias="none",
)

unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()
unet.enable_gradient_checkpointing()

print("✓ LoRA injected into UNet")

# Create datasets and dataloaders
print("Loading datasets...")

train_dataset = PennCampusDataset(
    csv_path="/content/drive/MyDrive/CIS_5190_Project_Material/penn_lora_train.csv",
    image_dir="/content/drive/MyDrive/CIS_5190_Project_Material/Images",
    edge_dir="/content/drive/MyDrive/CIS_5190_Project_Material/CannyMap",
    depth_dir="/content/drive/MyDrive/CIS_5190_Project_Material/Depth_Maps",
    tokenizer=tokenizer,
    resolution=RESOLUTION,
)

val_dataset = PennCampusDataset(
    csv_path="/content/drive/MyDrive/CIS_5190_Project_Material/penn_lora_val.csv",
    image_dir="/content/drive/MyDrive/CIS_5190_Project_Material/Images",
    edge_dir="/content/drive/MyDrive/CIS_5190_Project_Material/CannyMap",
    depth_dir="/content/drive/MyDrive/CIS_5190_Project_Material/Depth_Maps",
    tokenizer=tokenizer,
    resolution=RESOLUTION,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn,
)

print(f"\n{'='*70}")
print(f"Dataset ready:")
print(f"  Training: {len(train_dataset)} images ({len(train_loader)} batches)")
print(f"  Validation: {len(val_dataset)} images ({len(val_loader)} batches)")
print(f"  Total training steps: {len(train_loader) * NUM_EPOCHS}")
print(f"{'='*70}")

# Optimizer
trainable_params = [p for p in unet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE)

print(f"✓ Optimizer created with {len(trainable_params)} parameter groups")

# Training loop
print(f"\n{'='*70}")
print("Starting training...")
print(f"{'='*70}\n")

Loading model components...


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading Multi-ControlNet...
✓ Models loaded
trainable params: 1,594,368 || all params: 861,115,332 || trainable%: 0.1852
✓ LoRA injected into UNet
Loading datasets...
Dataset initialized with 95 valid samples
Dataset initialized with 16 valid samples

Dataset ready:
  Training: 95 images (24 batches)
  Validation: 16 images (4 batches)
  Total training steps: 360
✓ Optimizer created with 256 parameter groups

Starting training...



In [16]:
# Enable A100 optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

unet.train()
controlnet.eval()
global_step = 0
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

    for step, batch in enumerate(progress_bar):
        # Load data
        pixel_values = batch["pixel_values"].to(device, dtype=dtype, non_blocking=True)
        # --- FIX 1: Extract BOTH condition maps from the batch ---
        canny_cond = batch["canny_pixel_values"].to(device, dtype=dtype, non_blocking=True)
        depth_cond = batch["depth_pixel_values"].to(device, dtype=dtype, non_blocking=True)
        # ---------------------------------------------------------
        input_ids = batch["input_ids"].to(device, non_blocking=True)

        # Encode image to latent space
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

        # Sample noise and timesteps
        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        timesteps = torch.randint(
            0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device
        ).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # Encode text prompt
        with torch.no_grad():
            encoder_hidden_states = text_encoder(input_ids)[0]

        # Forward pass with ControlNet
        with torch.autocast("cuda", dtype=torch.bfloat16):
            # --- FIX 2: Pass both conditions as a list! ---
            down_block_res_samples, mid_block_res_sample = controlnet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=encoder_hidden_states,
                controlnet_cond=[canny_cond, depth_cond],
                conditioning_scale=[1.0,1.0],
                return_dict=False,
            )
            # ----------------------------------------------

            # UNet prediction with ControlNet conditioning
            noise_pred = unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states,
                down_block_additional_residuals=down_block_res_samples,
                mid_block_additional_residual=mid_block_res_sample,
            ).sample

            # MSE loss
            loss = F.mse_loss(noise_pred.float(), noise.float(), reduction="mean")

        # Backprop
        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

        # Track loss
        true_loss = loss.item()
        epoch_loss += true_loss
        progress_bar.set_postfix({"loss": f"{true_loss:.4f}"})

    avg_train_loss = epoch_loss / len(train_loader)

    # Validation
    if (epoch + 1) % VALIDATE_EVERY_N_EPOCHS == 0:
        val_loss = compute_validation_loss(
            unet, controlnet, val_loader, noise_scheduler,
            vae, text_encoder, device, dtype
        )

        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss:   {val_loss:.4f}")

        # Track best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            print(f"  ✓ New best validation loss!")

        # Generate visual samples
        generate_visual_validation(unet, controlnet, epoch + 1, VALIDATION_DIR)
    else:
        print(f"Epoch {epoch+1}/{NUM_EPOCHS}: Train Loss = {avg_train_loss:.4f}")

    # Save checkpoint
    if (epoch + 1) % SAVE_EVERY_N_EPOCH == 0:
        save_path = os.path.join(OUTPUT_DIR, f"lora_epoch_{epoch+1}")
        unet.save_pretrained(save_path)
        print(f"  ✓ Checkpoint saved: {save_path}")

# Final save
final_path = os.path.join(OUTPUT_DIR, "lora_final")
unet.save_pretrained(final_path)

print(f"\n{'='*70}")
print(f"Training complete!")
print(f"  Final model: {final_path}")
print(f"  Best validation loss: {best_val_loss:.4f}")
print(f"{'='*70}")

Epoch 1/15:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 1/15: Train Loss = 0.1254


Epoch 2/15:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 2/15: Train Loss = 0.1043


Epoch 3/15:   0%|          | 0/24 [00:00<?, ?it/s]

...


Epoch 15/15:
  Train Loss: 0.1397
  Val Loss:   0.1402
  Generating visual samples...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

  ✓ Checkpoint saved: /content/drive/MyDrive/CIS_5190_Project_Material/LoRA_Checkpoints(D+C)/lora_epoch_15

Training complete!
  Final model: /content/drive/MyDrive/CIS_5190_Project_Material/LoRA_Checkpoints(D+C)/lora_final
  Best validation loss: 0.0931
